In [1]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import random

In [2]:
!rm -rf /kaggle/working/utae-paps
!git clone https://github.com/VSainteuf/utae-paps.git
!pip install -q segmentation-models-pytorch
!touch /kaggle/working/utae-paps/src/__init__.py
!touch /kaggle/working/utae-paps/src/backbones/__init__.py

import sys
repo_path = '/kaggle/working/utae-paps'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("готово")

Cloning into 'utae-paps'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 97 (delta 34), reused 21 (delta 21), pack-reused 46 (from 1)
Receiving objects: 100% (97/97), 3.11 MiB | 27.26 MiB/s, done.
Resolving deltas: 100% (42/42), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00
готово


In [3]:
def pad_collate(batch):
    images, masks = zip(*batch)
    max_t = max([img.shape[0] for img in images])
    padded_images, batch_positions = [], []
    for img in images:
        t, c, h, w = img.shape
        pos = torch.arange(t) 
        if t < max_t:
            padding = torch.zeros((max_t - t, c, h, w))
            img = torch.cat([img, padding], dim=0)
            pos_padding = torch.zeros(max_t - t).long()
            pos = torch.cat([pos, pos_padding], dim=0)
        padded_images.append(img)
        batch_positions.append(pos)
    return torch.stack(padded_images), torch.stack(masks), torch.stack(batch_positions)

class PastisDataset(Dataset):
    def __init__(self, data_dir, patch_ids, augment=False):
        self.data_dir = data_dir
        self.patch_ids = patch_ids
        self.augment = augment

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        pid = self.patch_ids[idx]
        data_path = os.path.join(self.data_dir, 'DATA_S2', f'S2_{pid}.npy')
        
        # Проверяем оба варианта названия масок
        mask_path = os.path.join(self.data_dir, 'ANNOTATIONS', f'ParcelIDs_{pid}.npy')
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.data_dir, 'ANNOTATIONS', f'TARGET_{pid}.npy')

        data = np.load(data_path)
        mask = np.load(mask_path)
        if mask.ndim == 3: mask = mask[0]
        
        mask[mask > 19] = 0

        if self.augment:
            if random.random() > 0.5:
                data = np.flip(data, axis=3).copy(); mask = np.flip(mask, axis=1).copy()
            if random.random() > 0.5:
                data = np.flip(data, axis=2).copy(); mask = np.flip(mask, axis=0).copy()

        return torch.from_numpy(data).float(), torch.from_numpy(mask).long()

DATA_DIR = '/kaggle/input/datasets/khlaifiabilel/pastis/PASTIS'
raw_ids = [f.split('_')[1].split('.')[0] for f in os.listdir(os.path.join(DATA_DIR, 'DATA_S2')) if f.startswith('S2_')]

print(f"Всего снимков найдено: {len(raw_ids)}. Проверяем наличие масок...")

valid_ids = []
for pid in raw_ids:
    p1 = os.path.join(DATA_DIR, 'ANNOTATIONS', f'ParcelIDs_{pid}.npy')
    p2 = os.path.join(DATA_DIR, 'ANNOTATIONS', f'TARGET_{pid}.npy')
    if os.path.exists(p1) or os.path.exists(p2):
        valid_ids.append(pid)

print(f"Проверка окончена. Валидных пар (снимок + маска): {len(valid_ids)}")
if len(raw_ids) != len(valid_ids):
    print(f"Внимание! Удалено {len(raw_ids) - len(valid_ids)} битых записей без масок.")

np.random.seed(42)
np.random.shuffle(valid_ids)
split_idx = int(len(valid_ids) * 0.8)

train_ids = valid_ids[:split_idx]
test_ids = valid_ids[split_idx:]

train_loader = DataLoader(PastisDataset(DATA_DIR, train_ids, augment=True), 
                          batch_size=2,
                          shuffle=True, 
                          num_workers=0, 
                          collate_fn=pad_collate)

test_loader = DataLoader(PastisDataset(DATA_DIR, test_ids, augment=False), 
                         batch_size=2,
                         shuffle=False, 
                         num_workers=0, 
                         collate_fn=pad_collate)

Всего снимков найдено: 2468. Проверяем наличие масок...
Проверка окончена. Валидных пар (снимок + маска): 2433
Внимание! Удалено 35 битых записей без масок.


In [4]:
import gc

gc.collect()
torch.cuda.empty_cache()

from src.backbones.utae import UTAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UTAE(
    input_dim=10,
    encoder_widths=[64, 64, 64, 128],
    decoder_widths=[64, 64, 64, 128],
    out_conv=[32, 20],
    n_head=8,
    d_model=128,
    return_maps=False
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)

print(f"U-TAE на {device}")

U-TAE на cuda


In [ ]:
import sys

EPOCHS = 30
best_loss = float('inf')
save_path = '/kaggle/working/utae_pastis_best_v2.pth'

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Старт фонового обучения на {EPOCHS} эпох...")
print(f"Параметры: Device={device}, Batch Size={train_loader.batch_size}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    
    n_batches = len(train_loader)
    
    for batch_idx, (data, mask, pos) in enumerate(train_loader):
        data, mask, pos = data.to(device), mask.to(device), pos.to(device)
        
        optimizer.zero_grad()
        output = model(data, batch_positions=pos) 
        
        loss = criterion(output, mask)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
        with torch.no_grad():
            preds = torch.argmax(output, dim=1)
            correct_pixels += (preds == mask).sum().item()
            total_pixels += mask.numel()
        
        if batch_idx % 50 == 0 or batch_idx == n_batches - 1:
            curr_acc = (correct_pixels / total_pixels) * 100
            print(f"Epoch [{epoch}/{EPOCHS}] | Batch [{batch_idx}/{n_batches}] | Loss: {loss.item():.4f} | Acc: {curr_acc:.2f}%")
            sys.stdout.flush()
        
    scheduler.step()
    
    avg_loss = train_loss / n_batches
    avg_acc = (correct_pixels / total_pixels) * 100
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"\nИТОГ ЭПОХИ {epoch:02d}: Loss: {avg_loss:.4f} | Acc: {avg_acc:.2f}% | LR: {current_lr:.6f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), save_path)
        print(f"Новое достижение! Модель сохранена (Loss: {best_loss:.4f})")
    print("-" * 50)
    sys.stdout.flush()

print(f"\nОбучение полностью завершено. Веса здесь: {save_path}")

Старт фонового обучения на 30 эпох...
Параметры: Device=cuda, Batch Size=2
Epoch [1/30] | Batch [0/973] | Loss: 3.1916 | Acc: 7.43%
Epoch [1/30] | Batch [50/973] | Loss: 2.6304 | Acc: 47.40%
Epoch [1/30] | Batch [100/973] | Loss: 2.4793 | Acc: 53.67%
Epoch [1/30] | Batch [150/973] | Loss: 2.4377 | Acc: 64.87%
Epoch [1/30] | Batch [200/973] | Loss: 2.3977 | Acc: 73.11%
Epoch [1/30] | Batch [250/973] | Loss: 2.3586 | Acc: 76.62%
Epoch [1/30] | Batch [300/973] | Loss: 2.3162 | Acc: 78.73%
Epoch [1/30] | Batch [350/973] | Loss: 2.2517 | Acc: 80.88%
Epoch [1/30] | Batch [400/973] | Loss: 2.1995 | Acc: 82.87%
Epoch [1/30] | Batch [450/973] | Loss: 2.1486 | Acc: 84.54%
Epoch [1/30] | Batch [500/973] | Loss: 2.0999 | Acc: 86.00%
Epoch [1/30] | Batch [550/973] | Loss: 2.0510 | Acc: 87.25%
Epoch [1/30] | Batch [600/973] | Loss: 2.0044 | Acc: 88.09%
Epoch [1/30] | Batch [650/973] | Loss: 1.9590 | Acc: 88.79%
Epoch [1/30] | Batch [700/973] | Loss: 1.9137 | Acc: 89.55%
Epoch [1/30] | Batch [750/973